In [47]:
import os
from pathlib import Path
from uuid import uuid4

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma import Chroma

from src.md_reader import PreProcMd

OLLAMA_HOST = "127.0.0.1:11434"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
print(OLLAMA_HOST)

# ==========================================
EMBED_MODEL = "mxbai-embed-large:latest"
LLM_MODEL = "llama3.2:3b"
# ==========================================

127.0.0.1:11434


# Building Knowledge Base

## Loading document(s)
Here we'll load just a single document with classes from the [langchain-community](https://reference.langchain.com/python/langchain-community) document loader module. In the base example, we load a pdf file, but you may load pdf, txt, or markdown files. The loaders extract and load the files as [Document objects](https://reference.langchain.com/python/langchain-core/documents) defined in the [LangChain Core](https://reference.langchain.com/python/langchain-core) Python package.

In [24]:
def load_document(fpath: str):
    ext = os.path.splitext(fpath)[-1].lower()
    if ext == ".pdf":
        #mode: 'single' -> entire doc, (default) 'page' -> 1 doc/page
        loader = PyPDFLoader(fpath, mode="page")
        docs = loader.load()
    elif ext in [".md", ".markdown"]:
        repo_path = os.path.dirname(fpath)
        splits = PreProcMd(repo_path).run()
        target = os.path.abspath(fpath)
        docs = [
            d for d in splits
            if os.path.abspath(str(d.metadata.get("source", ""))) == target
        ]
    elif ext == ".txt":
        loader = TextLoader(fpath, encoding="utf-8")
        docs = loader.load()
    else:
        raise ValueError(f"Unsupported file type: {fpath}")
    print(f"Loaded {len(docs)} docs from {fpath}")
    return docs

In [25]:
docs_dir = '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs' 
file_name = 'data_machine.pdf'
file_path = os.path.join(docs_dir, file_name) #path to all doc: /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data
docs = load_document(file_path) #in 'page' mode number of docs here == number of pages in the actual document

Loaded 5 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf


In [26]:
docs[0]

Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='The Data Machine\n \nICER is excited to offer a new computational resource called the “Data Machine”. Many research \nareas are now faced with large amounts of data thanks in large part to the growth of available \ndatasets. Manipulating, analyzing, and visualizing  large amounts of data requires specialized \ncomputational resources that are not typically offered by more traditional high performance \ncomputing systems. Additionally, this “data explosion” is occurring primarily in fields where \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine is not ye

### Loading and preprocessing one Markdown document

The Docs Assistant Markdown reader separates YAML front matter and splits Markdown by level 1, 2, and 3 headings. The next cell uses those imported operations on one supplied Markdown file instead of scanning a directory.


In [27]:
# Import the Markdown preprocessing operations from Docs Assistant.
from src.md_reader import PreProcMd, extract_front
from src.kb import inject_md_context


def load_single_markdown(file_path: str) -> list[Document]:
    # Read only the supplied Markdown file.
    path = Path(file_path)
    text = path.read_text(encoding="utf-8", errors="replace")
    front_matter, body = extract_front(text)

    # Store the file information and front matter as document metadata.
    metadata = {"source": str(path), "doc_type": "markdown"}
    metadata.update(front_matter)

    tags = metadata.get("tags", [])
    if isinstance(tags, str):
        metadata["tags"] = [tags]
    elif not isinstance(tags, list):
        metadata["tags"] = []

    document = Document(page_content=body, metadata=metadata)

    # Apply the Docs Assistant heading splitter to this document.
    reader = PreProcMd(str(path.parent))
    return reader.splitdoc([document])


The next cell loads the Markdown file specified by `markdown_file_path`, adds its source and heading metadata to the embedded text, and displays the first chunk.


In [28]:
# Set this value to the path of one Markdown file.
markdown_file_path = Path("/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/public-documentation/docs/Data_Machine.md")

# Load and process only the supplied Markdown file.
markdown_docs = load_single_markdown(str(markdown_file_path))
markdown_docs = inject_md_context(markdown_docs)

print(f"Loaded {len(markdown_docs)} Markdown chunks.")
if markdown_docs:
    print(markdown_docs[0])


Loaded 18 Markdown chunks.
page_content='source: Data_Machine.md
heading: The Data Machine
tags: explanation, data machine

# The Data Machine  
ICER is excited to offer a new computational resource called the “Data Machine”. Many research areas are now faced with large amounts of data thanks in large part to the growth of available datasets. Manipulating, analyzing, and visualizing  large amounts of data requires specialized computational resources that are not typically offered by more traditional high performance computing systems. Additionally, this “data explosion” is occurring primarily in fields where research computing has historically not been widely used.  
To meet these hardware and educational needs, ICER is developing the Data Machine and associated outreach and training programs. Though the machine is not yet available to the broader research community (both at MSU and beyond), ICER is looking to connect with researchers who are interested in this new initiative and willi

### Extracting text from an image-based PDF

Image-based PDFs do not contain selectable text for `PyPDFLoader` to extract. The next cell renders each page as an image, applies Tesseract OCR, and returns one LangChain document per page containing text.


The next cell defines the PDF OCR operation. PyMuPDF renders each page, and Tesseract extracts the text from the rendered image.


In [29]:
def extract_pdf_image_text(pdf_path: str) -> list[Document]:
    import io
    import fitz
    import pytesseract
    from PIL import Image

    image_docs = []

    # Render each page and extract its text with OCR.
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            pixmap = page.get_pixmap()
            image = Image.open(io.BytesIO(pixmap.tobytes("png")))
            text = pytesseract.image_to_string(image)

            if text.strip():
                image_docs.append(
                    Document(
                        page_content=text,
                        metadata={"source": pdf_path, "page": page_number},
                    )
                )

    return image_docs


The next cell applies OCR to the PDF specified by `image_pdf_path` and displays the first extracted page.


In [30]:
# Set this value to the path of one image-based PDF.
image_pdf_path = Path("/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/test_img/img.pdf")

# Extract text from the supplied PDF.
image_docs = extract_pdf_image_text(str(image_pdf_path))

print(f"Created {len(image_docs)} OCR documents.")
if image_docs:
    print(image_docs[0])


Created 1 OCR documents.
page_content='go to a dev-node
_————

ssh dev-intel18

login gateway node

go to a dev-node
—_———

ssh dev-intell16

rsync gateway node

dev-intel18

dev-intel16-k80

dev-intel14-k20

©) submit jobs

dev-intel16

- §)

dev-intel14-phi

dev-intel14

A development node is mainly for:

-file navigation/editing
-software installation
-program testing (2-hr limit)
-job submission with sbatch

— ling, Clg
7g 2 rs
&\ ”

800+ compute nodes
(connected by high speed network)
' metadata={'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/test_img/img.pdf', 'page': 1}


## Splitting & Chunking Docs
We'll now take the loaded docs (list of Document objects), and split each of them into smaller chunks using the [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) from the [LangChain Text Splitters](https://reference.langchain.com/python/langchain-text-splitters) Python package. This applies splitting recursively based on typical splitting characters (e.g., ["\n\n", "\n", " ", ""]). The method first splits by paragraphs with "\n\n" and further splits each of those with "\n" (then space and empty string). Once the doc has been fully split, it then merges the split text to define chunks with maximum length determined by our set chunk size. 

The chunk size is essentially the maximum (and targeted) size of a chunk determined by length (len function by default). The chunk overlap is also a targeted value. Overlap helps maintain context when divided between chunks.

>Note the resulting chunk sizes and overlaps are often not exact, as these methods look for clean text boundaries (e.g., paragraph, line, word).  

In [31]:
def conditional_split(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """
    Conditionally apply chunking only to documents longer than chunk_size.
    Docs <= chunk_size pass through unchanged (1 chunk).
    Docs > chunk_size are split using RecursiveCharacterTextSplitter with overlap.

    Params:
        docs: Input list of Documents (typically markdown will be already structurally split).
        chunk_size: Target max characters per chunk.
        chunk_overlap: Number of overlapping characters between consecutive chunks.

    Returns:
        A list of Documents where longer inputs are expanded into multiple chunks.
    """
    splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True, separators=None, ) #default splitting characters ["\n\n", "\n", " ", ""]
    # splitter=CharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True,)
    out = []
    for i in docs:
        if len(i.page_content) <= chunk_size:
            out.append(i)  
        else:
            out.extend(splitter.split_documents([i]))  
    return out

In [32]:
# text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=1000, chunk_overlap=200, add_start_index=True
#     )
splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(splits)} chunks.")

Split into 21 chunks.


In [33]:
splits[0].page_content[-150:]

'ere \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and'

In [34]:
splits[1].page_content[:150]

'To meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine i'

## Document Embedding
Next, we will embed the documents and put them in a vector store collection. We'll use the [LangChain Ollama embedding integration](https://reference.langchain.com/python/langchain-ollama/embeddings/OllamaEmbeddings) to embed the docs with a local embedding model from our Ollama common model space. Here, we're using a [Chroma](https://docs.trychroma.com/docs/overview/introduction) vector store with the [LangChain Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma).

In [35]:
# embeddings = OllamaEmbeddings(model=EMBED_MODEL)
# vector_store = InMemoryVectorStore(embeddings)
# vector_store.add_documents(splits)
# print(f"{len(vector_store.store)} docs in vector store, one for each chunk.")

### Creating the vector store
The vector store is then created with name for our docs collection, and the `OllamaEmbeddings` instance to define/configure the embedding function. Note the `OllamaEmbeddings` class is used to store configuration data of the selected embedding model, and implements the embedding function using the selected embedding model. 

> In this case the vector store is setup and running in-memory, not on disk/external db.

> Run vector_store._client.delete_collection(collection_name) to remove collection from chroma vector store

In [36]:
collection_name = "icer_docs_collection"

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)

#vector_store._client.delete_collection("icer_docs_collection")

Now we add the documents to the vector store, providing unique ids to each document. The document embeddings are included in this step.

In [37]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids) # add_documents applies the embedding function to the documents (splits)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

42 docs in vector store, one for each chunk.
CPU times: user 24.7 ms, sys: 5.66 ms, total: 30.3 ms
Wall time: 345 ms


In [38]:
vs_embedding_info = vector_store._collection.get(include=["embeddings"])
doc_embeddings = vs_embedding_info['embeddings']

print(f"{doc_embeddings.shape[0]} doc embeddings of size {doc_embeddings.shape[1]} in vector store: \n\n", doc_embeddings)

42 doc embeddings of size 1024 in vector store: 

 [[ 0.02496761  0.00385572 -0.03172852 ... -0.02011916 -0.00144523
   0.04185246]
 [ 0.05354259  0.02565197 -0.05158573 ... -0.01262257 -0.01461829
   0.04328547]
 [ 0.02563222 -0.00211609  0.01004239 ...  0.00018149 -0.01497238
   0.06008345]
 ...
 [ 0.06278302 -0.01110651 -0.00132954 ... -0.07038857 -0.01094996
   0.01734013]
 [ 0.05116982 -0.02099884 -0.02040707 ... -0.03487178 -0.00247381
   0.02484726]
 [ 0.02855283 -0.00057942 -0.02564297 ... -0.0227016   0.00216078
   0.03079591]]


In [39]:
print(vs_embedding_info)

{'ids': ['557a9e21-22f3-4411-a292-3a4aeaad6944', '421bd4b4-2026-4e66-8fe6-59d11c551b88', 'd3ea18a4-6339-456c-bf14-2f6333637359', 'a291f690-17e7-4b98-80c1-02e40995ea2f', '0905171c-92d1-45a4-a2ac-5b2dca2e1cd2', '061b1738-ada4-4f63-bd71-69621c0e2d23', '7fbafb9a-927e-4301-a41c-93db6e6168af', '175c6643-acc3-417d-aa50-b30f3999d9d7', 'f73d2a83-5f8f-452f-a2fc-6f5c22af39ce', '627294ee-2445-4b09-a529-4e45b6dce8f4', '966b2396-0c96-40b4-8d76-98ed6c107e3e', '5bbb0400-f5cb-41bd-989e-cf7bcdecd748', '4e867e82-7367-441e-a0f7-823a7ac81bab', 'e122b359-32c5-4633-b903-a01c5b43def2', '107cb36f-50d5-4d86-8838-32ccb8b9450b', '813c4cff-6999-43da-a18f-b1c911f9fa02', '596add98-e5a1-4229-a305-0f5beaee92b4', 'c974a3c6-8fdf-43df-8a16-ec2bebe11523', '174fb405-8fc5-422f-87aa-d1be3e068f69', '6b07542f-c0dd-461a-ba51-c75328146fad', 'a11a3e33-ec58-4dec-b90b-4a96b834343e', '7741ad4a-4ced-40c3-ab27-0dcb7b42c968', 'c964efb9-17e8-4efb-9d44-b7fe65e56039', '659e2e58-3e3a-4fb4-8462-08f0cd468093', '7c2689da-adf5-491c-b167-a9521d

In [40]:
# vector_store._collection.get(include=["embeddings"]) #include ["documents", "metadatas", "embeddings"]

Here we run a similarity search for a simple query. The query is embedded and checked against the vector store embeddings, selecting the top k docs by similarity (or distance). Another method is using [`similarity_search_with_relevance_scores`](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/similarity_search_with_relevance_scores) which scores increasing with similarity.  

In [41]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.464797] {'total_pages': 5, 'creationdate': '2025-10-24T14:13:13+00:00', 'start_index': 2216, 'page_label': '2', 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'page': 1, 'creator': 'pdfmake', 'producer': 'pdfmake'} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.464797] {'start_index': 2216, 'creationdate': '2025-10-24T14:13:13+00:00', 'producer': 'pdfmake', 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'total_pages': 5, 'creator': 'pdfmake', 'page': 1, 'page_label': '2'}

In [42]:
results = vector_store.similarity_search_with_relevance_scores(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.671339] {'start_index': 2216, 'producer': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'page_label': '2', 'creator': 'pdfmake', 'total_pages': 5, 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'page': 1} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.671339] {'page': 1, 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'start_index': 2216, 'page_label': '2', 'producer': 'pdfmake', 'total_pages': 5, 'creationdate': '2025-10-24T14:13:13+00:00', 'creator': 'pdfmake'}

## Generating a JSON answer

The Docs Assistant retrieves relevant chunks, combines them as context, and passes a JSON schema to `ChatOllama`. The schema determines the fields returned by the model.


In [48]:
# Retrieve context for the question.
from langchain_core.prompts import ChatPromptTemplate

json_query = "What type of GPUs are on The Data Machine?"
json_results = vector_store.similarity_search_with_score(json_query, k=3)
json_context = "\n\n".join(doc.page_content for doc, _ in json_results)

# Define the JSON fields returned by the model.
json_schema = {
    "type": "object",
    "properties": {
        "topic": {"type": "string"},
        "summary": {"type": "string"},
        "details": {"type": "array", "items": {"type": "string"}},
        "commands": {"type": "array", "items": {"type": "string"}},
        "notes": {"type": "string"},
    },
    "required": ["topic", "summary"],
}

json_prompt = ChatPromptTemplate.from_template("""
You are a helpful faq/documentation assistant that answers questions using the provided document context.

<context>
{context}
</context>

Question: {query}

Answer clearly and concisely. Return only valid JSON matching the schema.
""")

# Pass the schema to the same chat model used by Docs Assistant.
json_llm = ChatOllama(model=LLM_MODEL, temperature=0, format=json_schema)
json_response = json_llm.invoke(
    json_prompt.format(context=json_context, query=json_query)
)
print(json_response.content)


{"topic": "GPU", "summary": "The document does not specify the exact model or type of GPUs used in The Data Machine.", "details": []}


## Building Knowledge Base
We'll now extend document embedding to a set of docs (pdfs in this example).

In [49]:
docs_repo = Path(docs_dir)
file_paths = list(docs_repo.glob("*.pdf"))
all_docs = []
for path in file_paths:
    all_docs.extend(load_document(path))

splits = conditional_split(all_docs, chunk_size=700, chunk_overlap=100)
print(f"Split into {len(splits)} chunks.")

Loaded 5 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf
Loaded 3 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/requesting_gpus.pdf
Loaded 3 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/cluster_resources.pdf
Loaded 5 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/llms_hpcc.pdf
Loaded 2 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/gpu_resources.pdf
Loaded 2 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/user_data_policies.pdf
Loaded 1 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/dev_nodes.pdf
Loaded 3 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/job_policies.pdf
Loaded 3 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/interacti

### Adding one Markdown document to the knowledge base

The next cell loads one supplied Markdown file, adds its metadata context, combines it with the loaded PDF documents, and applies the existing chunking operation.


In [50]:
# Set this value to the path of one Markdown file.
markdown_file_path = Path("/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/public-documentation/docs/Data_Machine.md")

# Add only the supplied Markdown document to the loaded documents.
markdown_docs = load_single_markdown(str(markdown_file_path))
markdown_docs = inject_md_context(markdown_docs)
all_docs.extend(markdown_docs)

# Recreate the chunks with the added Markdown document.
splits = conditional_split(all_docs, chunk_size=700, chunk_overlap=100)
print(f"Split into {len(splits)} chunks.")


Split into 176 chunks.


### Saving the vector store to a persistent directory

The next cells pass `persist_directory` to Chroma, add the document chunks, and then reopen the same collection from disk. Reopening the collection does not embed the stored documents again.

Rerunning the document-adding cell with new UUIDs adds duplicate chunks.


In [51]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get())

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [52]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

176 docs in vector store, one for each chunk.
CPU times: user 157 ms, sys: 37.4 ms, total: 195 ms
Wall time: 25.5 s


You can now query against a larger set of documents. As you can see, it may be nice to not have to run the embeddings each time you want to query against your documents, and instead query against the built and exisisting vector store. 

In [53]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.464797] {'creationdate': '2025-10-24T14:13:13+00:00', 'total_pages': 5, 'creator': 'pdfmake', 'page_label': '2', 'page': 1, 'start_index': 2216, 'producer': 'pdfmake', 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf'} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.483196] {'page': 2, 'creator': 'pdfmake', 'page_label': '3', 'producer': 'pdfmake', 'total_pages': 5, 'start_index': 531, 'source': '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/data_machine.pdf', 'creationdate': '2025-10-24T14:13:13+00:00'} 

Now we'll delete the vector store, and re-set to the persistent vector store db. We should see a collection already there. Now we do not need to re-apply the embedding function to our reference docs.

In [54]:
del vector_store

In [55]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get(include=["embeddings"]))

{'ids': ['3cdaf388-ef1b-4183-8570-a4fa33f4f14b', '2d57ad06-a99f-4161-9c6e-cec60c1b513b', 'f1d57672-0078-4433-a064-211cdac9840c', '7a8938ce-1fc1-43c4-9352-a552f8ec5d2f', 'b51d792d-97c8-4fce-b259-7e0c0f162a4f', '6e8ca9a9-6861-40f6-8e8f-f080078f0438', '2c796947-a99c-4cdc-b874-90651ab18132', 'bc3afbce-3008-4d4c-be5b-3c7564709d32', '54c96f60-3266-4b6e-8f3a-6b4035ff5534', 'b757b9e1-365c-484a-971f-164d2f466238', 'f5e0084b-1270-4a5d-93de-c821cd02f1a8', '215bc58b-b0b5-499c-aec0-acd27aa09b6e', '70274ffe-406e-42f5-a78c-70432b9b9599', 'af89d784-7aa7-4d63-9960-e95611760886', '72fe2697-84c1-4828-aecb-770f023e8ca2', 'f9b4370c-dfde-4aed-8326-133f2a2e95d9', '3b27e451-3d98-4205-acd2-c76dcf4357c6', 'fc405a52-7656-4059-8537-67c6b61c453b', 'a23e895f-d9dc-4a9a-8459-f5d60290f4e4', 'ec8a6f66-81df-4d88-9025-f55f9fd5ec79', '5409a879-ceb4-44c9-8c7d-5b8520ad3526', 'f2665b37-73b9-4980-9f01-d7133db0ef95', 'c995d7d9-a88a-4ba9-8f30-8c631b19fce2', '07a88300-c8b7-4526-b7ec-90064cc0c091', '13fdc914-ae70-487b-9c20-3a4542

### Adding documents to the persistent vector store

The next cells load and split another PDF, then add its chunks to the reopened Chroma collection. New UUIDs are generated each time the add cell runs.


In [57]:
file_path = '/mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/cluster_resources.pdf'
docs = load_document(file_path) 

Loaded 3 docs from /mnt/research/jdevenegas_research/icer_ai_projects/docs_bot/data/icer_docs/cluster_resources.pdf


In [58]:
new_splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(new_splits)} chunks.")

Split into 6 chunks.


In [59]:
uuids = [str(uuid4()) for _ in range(len(new_splits))]
vector_store.add_documents(documents=new_splits, ids=uuids) # add_documents applies the embedding function to the new documents (new splits)

['22100de2-ce7f-47f0-9662-55722e5ec5f1',
 '70f0687b-7e82-49e5-807f-15e4c1e6de4d',
 '00e234b2-1fe2-4d8c-986e-3ba131a45e0c',
 'a64d0684-f835-4be9-81f1-179b10b41857',
 'cbac24bb-59fe-408d-b004-01a002b291d0',
 'b4d5f6f7-22f4-4444-ac1f-be95859648cd']

## Benchmarking retrieval

The Docs Assistant benchmark loads a BEIR dataset containing documents, queries, and relevance labels. It builds the benchmark knowledge base with the same `build_knowledge_base` function used by the current Docs Assistant, then compares retrieved document IDs with the relevance labels.

The reported metrics are Precision@k, Recall@k, F1@k, Hit Rate@k, and MRR@k.


The next cell imports the current Docs Assistant functions used to load the dataset, build the knowledge base, and evaluate retrieval.


In [60]:
# Import the benchmark and knowledge-base operations from Docs Assistant.
from src.benchmark_runner import (
    load_beir_dataset,
    make_small_subset,
    corpus_to_documents,
    evaluate_retrieval,
)
from src.kb import build_knowledge_base


The next cell loads the SciFact test split and selects up to 500 documents and 25 queries while retaining the relevant documents.


In [63]:
# Load the benchmark documents, queries, and relevance labels.
corpus, queries, qrels = load_beir_dataset("scifact", split="test")

# Select the same small subset used by the current Docs Assistant benchmark.
corpus, queries, qrels = make_small_subset(
    corpus,
    queries,
    qrels,
    max_docs=100,
    max_queries=5,
)

print(f"Benchmark documents: {len(corpus)}")
print(f"Benchmark queries: {len(queries)}")


BEIR dataset already downloaded: scifact


  0%|          | 0/5183 [00:00<?, ?it/s]

Benchmark documents: 100
Benchmark queries: 5


The next cell converts the BEIR corpus to LangChain documents and passes them to the current Docs Assistant knowledge-base builder. That function splits the documents, embeds them, and stores them in an `InMemoryVectorStore`.


In [64]:
# Convert the BEIR corpus to LangChain documents.
benchmark_docs = corpus_to_documents(corpus)

# Build the benchmark store with the current Docs Assistant implementation.
benchmark_store = build_knowledge_base(
    docs=benchmark_docs,
    embed_model=EMBED_MODEL,
)


Split into 721 chunks.
Knowledge base ready.


The next cell runs retrieval for each query and reports the average scores at each value of `k`.


In [65]:
# Compare retrieved document IDs with the relevance labels.
evaluate_retrieval(
    vector_store=benchmark_store,
    queries=queries,
    qrels=qrels,
    k_values=[1, 3, 5, 10],
)



Total evaluated queries: 5

Scores @ 1
-----------------------------------
Precision@1: 0.8000
Recall@1:    0.7000
F1@1:        0.7333
HitRate@1:   0.8000
MRR@1:       0.8000

Scores @ 3
-----------------------------------
Precision@3: 0.4000
Recall@3:    1.0000
F1@3:        0.5600
HitRate@3:   1.0000
MRR@3:       0.9000

Scores @ 5
-----------------------------------
Precision@5: 0.2400
Recall@5:    1.0000
F1@5:        0.3810
HitRate@5:   1.0000
MRR@5:       0.9000

Scores @ 10
-----------------------------------
Precision@10: 0.1200
Recall@10:    1.0000
F1@10:        0.2121
HitRate@10:   1.0000
MRR@10:       0.9000


### Next steps

You can write a simple script to run an interactive chat with a docs chat assistant, which uses reference docs embedded in a vector store. You may use and/or reference the simple example we put in `scripts/docs_asst.py` and build on that, or test with your own set of reference docs. Note you can use the `--data_dir` flag to point the docs assistant to where your reference docs are stored, and it will apply an embedding on these docs. Note that if you do not provide specific files in that data directory, it will embed all docs. In either case, it will create these embeddings in-memory, and will not be persist between sessions (this is an example of something you could add). If you already have a persistent vector store directory (e.g., the chroma db you created in this notebook), you can simply point the docs assistant directly to the vector store using the `--persist_dir` flag. Another thing you may consider adding is the option to provide a persistent vector store db, as well as an additional set of docs which would be embedded and added to the existing (persistent) vector store. 

To run the `docs_asst.py` script with a full set of reference docs to be embedded, you would run  

`scripts/docs_asst.py --data_dir path/to/your/ref/docs`

To run the `docs_asst.py` script with a subset of reference docs to be embedded, you would run  

`scripts/docs_asst.py --data_dir path/to/your/ref/docs doc1.pdf doc2.pdf doc3.pdf`

To run the `docs_asst.py` script with a persistent vector store of your reference docs, you would run  

`scripts/docs_asst.py --persist_dir path/to/your/vector/store`

> Notes: (1) The LLM model for the docs assistant and embedding model for the vector store embeddings are hard coded as global variables in the `docs_asst.py` script as EMBED_MODEL and LLM_MODEL. You can modify these, or add a flag/option to the script to set your desired models accordingly. (2) Also, if using the `--persist_dir` flag it is currently setup and hard coded to use the `icer_docs_collection_full` collection that was created in this notebook. If you want to change this for a new set of reference docs you've embedded in a persistent vector store db, you can modify the script to use your new collection. 